# Feature engineering

Para la preparación del conjunto de datos que se utilizará para la clasificación se decidió que se prepararán dos para cada tipo de lípido (`CHOL` y `DPSM`).
Con estos dos conjuntos se entrenarán dos modelos para analizar si cada uno puede diferenciar entre los grupos **Mouse** y **NMR** para los dos tipos de lípido `CHOL` y `DPSM`.

Para cada molécula que se encuentra en `data` se obtendrán las siguientes características:
* Residual: Para la $i$-ésima molécula obtener su residual.
* Magnitud: Obtener media y desviación estándar.
* Proporciones: Obtener el cociente entre las medias de `head` y `body` o `heads` y `tails` dependiendo del tipo de lípido. Igualmente para las desviaciones estándar.
* Correlación: El coeficiente $r$ de Pearson entre `head` y `body` o `heads` y `tails` dependiendo del tipo de lípido.
* Amplitud: Valores de rango, mínimo y máximo valor.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

## Cargar datos

In [2]:
%%time
# Estructura de datos
data: dict[str, dict[str, np.ndarray]] = {}

target_folders = ["Mouse", "NMR"]

# Extraer archivos .npy
for folder_name in target_folders:
    sub_folder = Path("..")  / folder_name

    if sub_folder.is_dir():
        data[sub_folder.name] = {
            file.stem: np.load(file) for file in sub_folder.glob("*.npy")
        }

# Obtener información sobre valores nulos, valores infinitos, tipo de dato y forma
for d in data:
    for lipid in data[d]:
        array = data[d][lipid]
        nans = np.isnan(array).sum()
        infs = np.isinf(array).sum()
        dtype = array.dtype
        shape = array.shape
        print(
            f"Array {d} {lipid}: nans={nans}, infs={infs}, dtype={dtype}, shape={shape}"
        )

Array Mouse g3_DPSM_heads_resids: nans=0, infs=0, dtype=int64, shape=(342,)
Array Mouse g3_DPSM_tails: nans=0, infs=0, dtype=float64, shape=(342, 401, 201)
Array Mouse artificial_volume: nans=0, infs=0, dtype=float64, shape=(1,)
Array Mouse g3_DPSM_heads: nans=0, infs=0, dtype=float64, shape=(342, 401, 201)
Array Mouse g3_CHOL_head: nans=0, infs=0, dtype=float64, shape=(170, 401, 201)
Array Mouse g3_CHOL_head_resids: nans=0, infs=0, dtype=int64, shape=(170,)
Array Mouse g3_DPSM_tails_resids: nans=0, infs=0, dtype=int64, shape=(342,)
Array Mouse g3_CHOL_body_resids: nans=0, infs=0, dtype=int64, shape=(170,)
Array Mouse g3_CHOL_body: nans=0, infs=0, dtype=float64, shape=(170, 401, 201)
Array NMR g3_DPSM_heads_resids: nans=0, infs=0, dtype=int64, shape=(170,)
Array NMR g3_DPSM_tails: nans=0, infs=0, dtype=float64, shape=(170, 401, 201)
Array NMR artificial_volume: nans=0, infs=0, dtype=float64, shape=(1,)
Array NMR g3_DPSM_heads: nans=0, infs=0, dtype=float64, shape=(170, 401, 201)
Array 

In [3]:
%%time
chol_feats, dpsm_feats = [], []

for group in ("Mouse", "NMR"):
    for lipid_type in ("CHOL", "DPSM"):

        if lipid_type == "CHOL":
            head_data = data[group]["g3_CHOL_head"]
            body_data = data[group]["g3_CHOL_body"]
            head_resids = data[group]["g3_CHOL_head_resids"]
            body_resids = data[group]["g3_CHOL_body_resids"]

            n_molecules = head_data.shape[0]
            for i in range(n_molecules):

                mean_head = np.mean(head_data[i, :, :])
                mean_body = np.mean(body_data[i, :, :])
                std_head = np.std(head_data[i, :, :])
                std_body = np.std(body_data[i, :, :])
                min_head = np.min(head_data[i, :, :])
                max_head = np.max(head_data[i, :, :])
                min_body = np.min(body_data[i, :, :])
                max_body = np.max(body_data[i, :, :])

                dict_feats = {
                    "resid": head_resids[i],
                    "lipid_type": lipid_type,
                    "mean_head": mean_head,
                    "mean_body": mean_body, 
                    "std_head": std_head,
                    "std_body": std_body,
                    "ratio_head_body": mean_head / mean_body,
                    "ratio_std_head_body": std_head / std_body,
                    "corr_head_body": np.corrcoef(head_data[i, :, :].flatten(), body_data[i, :, :].flatten())[0, 1],
                    "min_head": min_head,
                    "max_head": max_head,
                    "range_head": max_head - min_head,
                    "min_body": min_body,
                    "max_body": max_body,
                    "range_body": max_body - min_body,
                    "target": 0 if group == "Mouse" else 1
                }

                chol_feats.append(dict_feats)
        else:  # DPSM
            heads_data = data[group]["g3_DPSM_heads"]
            tails_data = data[group]["g3_DPSM_tails"]
            heads_resids = data[group]["g3_DPSM_heads_resids"]
            tails_resids = data[group]["g3_DPSM_tails_resids"]

            n_molecules = heads_data.shape[0]
            for i in range(n_molecules):

                mean_heads = np.mean(heads_data[i, :, :])
                mean_tails = np.mean(tails_data[i, :, :])
                std_heads = np.std(heads_data[i, :, :])
                std_tails = np.std(tails_data[i, :, :])
                min_heads = np.min(heads_data[i, :, :])
                max_heads = np.max(heads_data[i, :, :])
                min_tails = np.min(tails_data[i, :, :])
                max_tails = np.max(tails_data[i, :, :])

                dict_feats = {
                    "resid": heads_resids[i],
                    "lipid_type": lipid_type,
                    "mean_heads": mean_heads,
                    "mean_tails": mean_tails,
                    "std_heads": std_heads,
                    "std_tails": std_tails,
                    "ratio_heads_tails": mean_heads / mean_tails,
                    "ratio_std_heads_tails": std_heads / std_tails,
                    "corr_heads_tails": np.corrcoef(heads_data[i, :, :].flatten(), tails_data[i, :, :].flatten())[0, 1],
                    "min_heads": min_heads,
                    "max_heads": max_heads,
                    "range_heads": max_heads - min_heads,
                    "min_tails": min_tails,
                    "max_tails": max_tails,
                    "range_tails": max_tails - min_tails,
                    "target": 0 if group == "Mouse" else 1
                }

                dpsm_feats.append(dict_feats)

CPU times: user 8.29 s, sys: 626 ms, total: 8.91 s
Wall time: 1.2 s


In [4]:
df_chol = pd.DataFrame(chol_feats)
df_dpsm = pd.DataFrame(dpsm_feats)

In [5]:
df_chol.columns

Index(['resid', 'lipid_type', 'mean_head', 'mean_body', 'std_head', 'std_body',
       'ratio_head_body', 'ratio_std_head_body', 'corr_head_body', 'min_head',
       'max_head', 'range_head', 'min_body', 'max_body', 'range_body',
       'target'],
      dtype='str')

In [6]:
df_dpsm.columns

Index(['resid', 'lipid_type', 'mean_heads', 'mean_tails', 'std_heads',
       'std_tails', 'ratio_heads_tails', 'ratio_std_heads_tails',
       'corr_heads_tails', 'min_heads', 'max_heads', 'range_heads',
       'min_tails', 'max_tails', 'range_tails', 'target'],
      dtype='str')

## Exportación de datos

In [7]:
df_chol.to_csv("../data/chol.csv", index=False)
df_dpsm.to_csv("../data/dpsm.csv", index=False)